# Sarcastic Conversational Dataset Generator – 6,000 Q&A Pairs

This notebook uses NVIDIA NIM's chat-completions API to create 6,000 unique, category-balanced sarcastic conversation examples. It validates each response, maintains category and sarcasm-level quotas, checkpoints accepted records to Google Drive, and resumes safely after a Colab disconnect.

**Cost and runtime:** generation makes real API requests and can incur NVIDIA API usage. It is not run automatically by opening the notebook; execute cells in order and provide your own NVIDIA API key through Colab Secrets or the hidden prompt. Never share the key.

Initial requests target 100 examples. Smaller adaptive requests are used after truncated/incomplete responses and near the end. API failures stop with a useful error while preserving the last checkpoint.

## 2. Install dependencies

Install the small set of notebook dependencies.

In [ ]:
%pip -q install requests pandas tqdm


## 3. Imports and configuration

The endpoint, model, generation settings, quotas, retry policy, and paths are centralized here.

In [ ]:
import csv
import difflib
import json
import logging
import math
import os
import random
import re
import tempfile
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

INVOKE_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
MODEL = "mistralai/mistral-large-4-1t-a52b"
STREAM = False
MAX_TOKENS = 4096
TEMPERATURE = 0.5
TOP_P = 1
FREQUENCY_PENALTY = 0
PRESENCE_PENALTY = 0
REASONING_EFFORT = "high"

TARGET_SIZE = 6_000
BATCH_SIZE = 100
MIN_BATCH_SIZE = 1
MAX_RETRIES = 6
BACKOFF_BASE_SECONDS = 2
BACKOFF_MAX_SECONDS = 60
CONNECT_TIMEOUT_SECONDS = 30
READ_TIMEOUT_SECONDS = 300
RANDOM_SEED = 2026
MAX_EMPTY_RESPONSES = 12

CATEGORY_QUOTAS = {
    "everyday_conversations": 500,
    "obvious_and_silly_questions": 450,
    "workplace_and_professional_situations": 450,
    "friends_and_casual_conversations": 500,
    "relationships_and_dating": 400,
    "technology_and_programming": 450,
    "school_college_and_education": 450,
    "annoying_everyday_situations": 450,
    "self_deprecating_humor": 350,
    "dry_humor_and_deadpan_sarcasm": 450,
    "clever_comebacks_and_witty_replies": 500,
    "internet_culture_and_social_media": 350,
    "overconfident_and_arrogant_statements": 300,
    "unexpected_situations_and_irony": 400,
}

assert sum(CATEGORY_QUOTAS.values()) == TARGET_SIZE
assert all(quota % 5 == 0 for quota in CATEGORY_QUOTAS.values())
LEVELS = (1, 2, 3, 4, 5)
LEVEL_QUOTAS = {
    category: {level: quota // len(LEVELS) for level in LEVELS}
    for category, quota in CATEGORY_QUOTAS.items()
}
CATEGORY_DESCRIPTIONS = {
    "everyday_conversations": "ordinary routines, small talk, plans, food, errands, and daily choices",
    "obvious_and_silly_questions": "deliberately silly or self-evident questions answered with a fresh twist",
    "workplace_and_professional_situations": "meetings, email, deadlines, office norms, and professional mishaps",
    "friends_and_casual_conversations": "specific, warm banter between friends in informal settings",
    "relationships_and_dating": "dates, communication, affection, and relationship quirks; keep it respectful",
    "technology_and_programming": "devices, software, debugging, apps, and developer experiences",
    "school_college_and_education": "classes, studying, exams, teachers, and campus life",
    "annoying_everyday_situations": "queues, traffic, chores, delays, minor inconveniences, and awkward timing",
    "self_deprecating_humor": "lighthearted first-person flaws and mishaps without endorsing self-harm",
    "dry_humor_and_deadpan_sarcasm": "understated, concise, emotionally restrained irony",
    "clever_comebacks_and_witty_replies": "a pointed, context-aware reply to a distinct remark or claim",
    "internet_culture_and_social_media": "feeds, posts, trends, notifications, and online habits; slang only when natural",
    "overconfident_and_arrogant_statements": "comically overconfident harmless claims met with a witty deflation",
    "unexpected_situations_and_irony": "reversals, coincidences, and outcomes that contradict expectations",
}

DRIVE_DIR = Path("/content/drive/MyDrive/Sarcasm_Dataset")
DATA_DIR = DRIVE_DIR
CHECKPOINT_PATH = DRIVE_DIR / "generation_checkpoint.json"
LOG_PATH = DRIVE_DIR / "generation.log"
OUTPUT_PATHS = {
    "jsonl": DRIVE_DIR / "sarcasm_dataset_6000.jsonl",
    "csv": DRIVE_DIR / "sarcasm_dataset_6000.csv",
    "train": DRIVE_DIR / "sarcasm_chat_training.jsonl",
    "validation": DRIVE_DIR / "sarcasm_validation.jsonl",
    "report": DRIVE_DIR / "generation_report.txt",
}

print(f"Model: {MODEL} | target: {TARGET_SIZE:,} | initial batch size: {BATCH_SIZE}")


## 4. Google Drive integration

Mount Drive and create the persistent dataset folder. Checkpoints and exports are written here.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Persistent output directory: {DRIVE_DIR}")


## 5. NVIDIA API authentication

Add `NVIDIA_API_KEY` under Colab Secrets (key icon), or enter it at the hidden fallback prompt. The key is never printed or written to Drive.

In [ ]:
NVIDIA_API_KEY = None
try:
    from google.colab import userdata

    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    NVIDIA_API_KEY = None

if not NVIDIA_API_KEY:
    from getpass import getpass

    NVIDIA_API_KEY = getpass("Enter NVIDIA API key (input is hidden): ").strip()
if not NVIDIA_API_KEY:
    raise RuntimeError("NVIDIA_API_KEY is required. Add it to Colab Secrets or enter it at the hidden prompt.")

print("NVIDIA API key found; its value is not displayed.")


## 6. NVIDIA API client

Use the specified NIM endpoint/model and generation settings. Retry rate limits, transient server errors, and network failures with exponential backoff.

In [ ]:
class NVIDIAAPIError(RuntimeError):
    pass


def save_checkpoint(state):
    """Atomically persist all accepted records and generation counters."""
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    payload = dict(state)
    payload["updated_at"] = datetime.now(timezone.utc).isoformat()
    fd, temp_name = tempfile.mkstemp(prefix="checkpoint_", suffix=".tmp", dir=DRIVE_DIR)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as temp_file:
            json.dump(payload, temp_file, ensure_ascii=False, indent=2)
            temp_file.flush()
            os.fsync(temp_file.fileno())
        os.replace(temp_name, CHECKPOINT_PATH)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)


STATE = None
LOGGER = logging.getLogger("sarcasm_generator")


def nvidia_chat_completion(messages, max_tokens=MAX_TOKENS):
    headers = {
        "Authorization": f"Bearer {NVIDIA_API_KEY}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "stream": STREAM,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "frequency_penalty": FREQUENCY_PENALTY,
        "presence_penalty": PRESENCE_PENALTY,
        "reasoning_effort": REASONING_EFFORT,
    }
    last_error = None
    for attempt in range(MAX_RETRIES):
        if STATE is not None:
            STATE["api_requests"] += 1
            save_checkpoint(STATE)
        retry_after = None
        try:
            response = requests.post(
                INVOKE_URL,
                headers=headers,
                json=payload,
                timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
            )
            if response.status_code == 429 or 500 <= response.status_code < 600:
                retry_after = response.headers.get("Retry-After")
                response.raise_for_status()
            response.raise_for_status()
            try:
                body = response.json()
                content = body["choices"][0]["message"]["content"]
                finish_reason = body["choices"][0].get("finish_reason")
            except (ValueError, KeyError, IndexError, TypeError) as exc:
                raise NVIDIAAPIError(f"Malformed NVIDIA API response: {exc}") from exc
            if not isinstance(content, str):
                raise NVIDIAAPIError("NVIDIA API returned non-text message content.")
            return content, finish_reason
        except requests.HTTPError as exc:
            status = exc.response.status_code if exc.response is not None else None
            detail = ""
            if exc.response is not None:
                detail = exc.response.text[:1000]
            last_error = NVIDIAAPIError(f"NVIDIA API HTTP error {status}: {detail}")
            retryable = status in (408, 425, 429) or (status is not None and 500 <= status < 600)
            if not retryable:
                raise last_error from exc
        except (requests.Timeout, requests.ConnectionError) as exc:
            last_error = NVIDIAAPIError(f"Temporary NVIDIA API network error: {exc}")
        except NVIDIAAPIError as exc:
            last_error = exc
            if attempt + 1 == MAX_RETRIES:
                raise

        if attempt + 1 < MAX_RETRIES:
            try:
                delay = min(float(retry_after), BACKOFF_MAX_SECONDS) if retry_after else None
            except ValueError:
                delay = None
            if delay is None:
                delay = min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)
            LOGGER.warning("NVIDIA request failed; retry %d/%d in %.1f seconds: %s",
                           attempt + 1, MAX_RETRIES - 1, delay, last_error)
            time.sleep(delay)
    raise NVIDIAAPIError(f"NVIDIA API retries exhausted: {last_error}") from last_error


## 7. Dataset generation prompt

Each request specifies exact category/intensity slots. The model must return JSONL only, with distinct scenarios and non-harmful humor.

In [ ]:
SYSTEM_PROMPT = """You write original, natural conversational Q&A examples for a sarcastic assistant.
Return ONLY JSONL: exactly one complete JSON object per line, no markdown, no code fences,
no prose before or after. Each object must have exactly these fields:
{"category":"...","sarcasm_level":1,"question":"...","answer":"..."}
Follow the supplied category and sarcasm-level counts exactly. Do not emit IDs.
Questions and answers must be contextually paired; answers should sound spontaneous,
intelligent, human, and entertaining. Vary length (short one-liners and fuller replies),
openings, syntax, vocabulary, point of view, and punchline placement. Use distinct
situations, not paraphrases. Include natural questions and statements as prompts.
Match intensity: 1 mild/friendly, 2 playful, 3 clearly witty/ironic, 4 sharp but clever,
5 exaggerated comedic/savage while still playful. Do not make every answer intense.
Avoid recycled or generic punchlines, repeated sentence templates, excessive "Oh really?",
"Yeah, right", or "Wow, impressive", and irrelevant sarcasm. Use slang only where natural.
Keep teasing non-abusive. Never include hateful, discriminatory, threatening, or abusive
content; no slurs, protected-class attacks, or encouragement of self-harm."""


def build_generation_prompt(plan, existing_questions):
    quotas = [
        {
            "category": category,
            "category_description": CATEGORY_DESCRIPTIONS[category],
            "sarcasm_level": level,
            "count": count,
        }
        for (category, level), count in plan.items()
    ]
    recent = existing_questions[-80:]
    examples_to_avoid = "\n".join(f"- {question}" for question in recent) or "(none yet)"
    return (
        "Generate exactly the requested number of JSONL records. Use only these slots; "
        "never exceed any listed category/intensity count. Keep question and answer text "
        "as single-line JSON strings (escape embedded newlines). Make every pair novel.\n\n"
        f"Requested category/intensity counts:\n{json.dumps(quotas, ensure_ascii=False)}\n\n"
        f"Questions already used; avoid these topics and wording:\n{examples_to_avoid}"
    )


## 8. Batch generation engine

The engine targets 100 examples per request, follows remaining quotas, assigns sequential IDs locally, and requests replacements for missing, rejected, or duplicate items.

In [ ]:
REQUIRED_FIELDS = {"id", "category", "sarcasm_level", "question", "answer"}
MODEL_FIELDS = {"category", "sarcasm_level", "question", "answer"}


def remaining_slots(state):
    counts = state["category_counts"]
    levels = state["level_counts"]
    slots = {}
    for category in CATEGORY_QUOTAS:
        for level in LEVELS:
            remaining = LEVEL_QUOTAS[category][level] - levels[category][str(level)]
            if remaining > 0:
                slots[(category, level)] = remaining
    return slots


def build_plan(state, limit):
    remaining = remaining_slots(state)
    plan = defaultdict(int)
    for _ in range(min(limit, sum(remaining.values()))):
        available = [key for key, count in remaining.items() if count > 0]
        if not available:
            break
        category, level = min(
            available,
            key=lambda key: (
                state["level_counts"][key[0]][str(key[1])] / LEVEL_QUOTAS[key[0]][key[1]],
                list(CATEGORY_QUOTAS).index(key[0]),
                key[1],
            ),
        )
        plan[(category, level)] += 1
        remaining[(category, level)] -= 1
    return dict(plan)


def parse_jsonl_response(content):
    text = content.strip()
    if text.startswith("```"):
        text = re.sub(r"\A```(?:jsonl|json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```\Z", "", text)
    records, invalid = [], 0
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            invalid += 1
            continue
        if not isinstance(item, dict):
            invalid += 1
            continue
        records.append(item)
    return records, invalid


def new_generation_state():
    return {
        "version": 1,
        "model": MODEL,
        "target_size": TARGET_SIZE,
        "records": [],
        "next_id": 1,
        "category_counts": {category: 0 for category in CATEGORY_QUOTAS},
        "level_counts": {
            category: {str(level): 0 for level in LEVELS}
            for category in CATEGORY_QUOTAS
        },
        "completed_batches": 0,
        "api_requests": 0,
        "duplicate_rejections": 0,
        "invalid_records": 0,
        "surplus_records": 0,
        "adaptive_batch_size": BATCH_SIZE,
        "consecutive_empty_responses": 0,
    }


def generate_dataset(state):
    progress = tqdm(total=TARGET_SIZE, initial=len(state["records"]), desc="Unique Q&A pairs")
    try:
        while len(state["records"]) < TARGET_SIZE:
            plan = build_plan(state, min(state["adaptive_batch_size"], BATCH_SIZE))
            if not plan:
                raise RuntimeError("No remaining quota slots, but the dataset is incomplete.")
            requested = sum(plan.values())
            prompt = build_generation_prompt(
                plan,
                [record["question"] for record in state["records"]],
            )
            content, finish_reason = nvidia_chat_completion(
                [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ]
            )
            candidates, malformed = parse_jsonl_response(content)
            state["invalid_records"] += malformed
            expected_by_slot = dict(plan)
            accepted = 0
            for item in candidates:
                if not MODEL_FIELDS.issubset(item) or set(item) - (MODEL_FIELDS | {"id"}):
                    state["invalid_records"] += 1
                    continue
                category = item.get("category")
                level = item.get("sarcasm_level")
                question = item.get("question")
                answer = item.get("answer")
                if (
                    not isinstance(category, str)
                    or category not in CATEGORY_QUOTAS
                    or type(level) is not int
                    or level not in LEVELS
                    or not isinstance(question, str)
                    or not isinstance(answer, str)
                ):
                    state["invalid_records"] += 1
                    continue
                question = re.sub(r"\s+", " ", question).strip()
                answer = re.sub(r"\s+", " ", answer).strip()
                if not question or not answer or len(question) > 2_000 or len(answer) > 3_000:
                    state["invalid_records"] += 1
                    continue
                slot = (category, level)
                if expected_by_slot.get(slot, 0) <= 0:
                    state["invalid_records"] += 1
                    continue
                record = {
                    "id": state["next_id"],
                    "category": category,
                    "sarcasm_level": level,
                    "question": question,
                    "answer": answer,
                }
                duplicate_kind = duplicate_detector.check_and_add(record)
                if duplicate_kind:
                    state["duplicate_rejections"] += 1
                    continue
                expected_by_slot[slot] -= 1
                state["records"].append(record)
                state["next_id"] += 1
                state["category_counts"][category] += 1
                state["level_counts"][category][str(level)] += 1
                accepted += 1
                progress.update(1)

            missing = sum(expected_by_slot.values())
            truncated = finish_reason == "length" or len(candidates) < requested
            if truncated:
                state["adaptive_batch_size"] = max(
                    MIN_BATCH_SIZE,
                    min(state["adaptive_batch_size"], max(MIN_BATCH_SIZE, math.ceil(requested / 2))),
                )
            elif state["adaptive_batch_size"] < BATCH_SIZE:
                state["adaptive_batch_size"] = min(BATCH_SIZE, state["adaptive_batch_size"] * 2)

            state["invalid_records"] += max(0, requested - len(candidates) - malformed)
            state["surplus_records"] += max(0, len(candidates) - requested)
            if accepted:
                state["completed_batches"] += 1
            if accepted == 0:
                state["consecutive_empty_responses"] += 1
            else:
                state["consecutive_empty_responses"] = 0
            save_checkpoint(state)
            show_progress(state, progress.n)
            if state["consecutive_empty_responses"] >= MAX_EMPTY_RESPONSES:
                raise RuntimeError(
                    "Too many consecutive responses produced no accepted examples. "
                    "Checkpoint is saved; inspect the API/model response and rerun to resume."
                )
            if missing and not candidates:
                LOGGER.warning("No parseable records in response; reducing request size to %d.",
                               state["adaptive_batch_size"])
    finally:
        progress.close()
    return state


## 9. Duplicate detection

Exact normalized questions and Q&A pairs are rejected. A lightweight character-ngram index flags near-identical questions without external embedding models.

In [ ]:
def normalize_text(value):
    value = unicodedata.normalize("NFKC", value).casefold()
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)
    return re.sub(r"\s+", " ", value).strip()


def character_ngrams(value, n=4):
    compact = f"  {normalize_text(value)}  "
    return {compact[index:index + n] for index in range(max(0, len(compact) - n + 1))}


class DuplicateDetector:
    def __init__(self, records=()):
        self.questions = set()
        self.pairs = set()
        self.question_grams = {}
        self.gram_index = defaultdict(set)
        for record in records:
            self._insert(record)

    def _insert(self, record):
        question = normalize_text(record["question"])
        answer = normalize_text(record["answer"])
        self.questions.add(question)
        self.pairs.add((question, answer))
        grams = character_ngrams(question)
        self.question_grams[question] = grams
        for gram in grams:
            self.gram_index[gram].add(question)

    def check_and_add(self, record):
        question = normalize_text(record["question"])
        answer = normalize_text(record["answer"])
        if question in self.questions:
            return "question"
        if (question, answer) in self.pairs:
            return "pair"
        grams = character_ngrams(question)
        if len(question) >= 24 and grams:
            overlap = Counter(
                prior
                for gram in grams
                for prior in self.gram_index.get(gram, ())
            )
            candidates = [
                prior for prior, shared in overlap.items()
                if shared / max(1, min(len(grams), len(self.question_grams[prior]))) >= 0.72
            ]
            for prior in candidates:
                prior_grams = self.question_grams[prior]
                union = len(grams | prior_grams)
                similarity = len(grams & prior_grams) / union if union else 1.0
                if similarity >= 0.90 or difflib.SequenceMatcher(None, question, prior).ratio() >= 0.94:
                    return "near_question"
        self._insert(record)
        return None


duplicate_detector = None


## 10. Checkpoint and resume

The checkpoint stores records, next ID, quota counts, successful batches, API requests, rejection counters, and adaptive batch size. Choose resume or start new; starting new requires explicit confirmation.

In [ ]:
def validate_checkpoint_shape(state):
    if state.get("version") != 1 or state.get("model") != MODEL or state.get("target_size") != TARGET_SIZE:
        raise RuntimeError("Checkpoint version/model/target does not match this notebook configuration.")
    records = state.get("records")
    if not isinstance(records, list) or len(records) > TARGET_SIZE:
        raise RuntimeError("Checkpoint has invalid records.")
    if state.get("next_id") != len(records) + 1:
        raise RuntimeError("Checkpoint next_id does not match its saved records.")
    expected_categories = {category: 0 for category in CATEGORY_QUOTAS}
    expected_levels = {
        category: {str(level): 0 for level in LEVELS}
        for category in CATEGORY_QUOTAS
    }
    saved_detector = DuplicateDetector()
    for index, record in enumerate(records, start=1):
        if not isinstance(record, dict) or set(record) != REQUIRED_FIELDS:
            raise RuntimeError(f"Checkpoint record {index} has an invalid schema.")
        if (
            type(record["id"]) is not int
            or record["id"] != index
            or not isinstance(record["category"], str)
            or record["category"] not in CATEGORY_QUOTAS
        ):
            raise RuntimeError(f"Checkpoint record {index} has an invalid ID or category.")
        if type(record["sarcasm_level"]) is not int or record["sarcasm_level"] not in LEVELS:
            raise RuntimeError(f"Checkpoint record {index} has an invalid sarcasm level.")
        if not isinstance(record["question"], str) or not record["question"].strip():
            raise RuntimeError(f"Checkpoint record {index} has an empty question.")
        if not isinstance(record["answer"], str) or not record["answer"].strip():
            raise RuntimeError(f"Checkpoint record {index} has an empty answer.")
        if saved_detector.check_and_add(record):
            raise RuntimeError(f"Checkpoint record {index} duplicates an earlier question.")
        expected_categories[record["category"]] += 1
        expected_levels[record["category"]][str(record["sarcasm_level"])] += 1
    if state.get("category_counts") != expected_categories or state.get("level_counts") != expected_levels:
        raise RuntimeError("Checkpoint category or sarcasm-level counters do not match its records.")
    if any(expected_categories[category] > quota for category, quota in CATEGORY_QUOTAS.items()):
        raise RuntimeError("Checkpoint exceeds a category quota.")
    if any(
        expected_levels[category][str(level)] > LEVEL_QUOTAS[category][level]
        for category in CATEGORY_QUOTAS
        for level in LEVELS
    ):
        raise RuntimeError("Checkpoint exceeds a sarcasm-level quota.")
    for key in ("api_requests", "duplicate_rejections", "invalid_records", "completed_batches"):
        if type(state.get(key)) is not int or state[key] < 0:
            raise RuntimeError(f"Checkpoint counter {key!r} is invalid.")
    return state


def load_checkpoint():
    try:
        with CHECKPOINT_PATH.open("r", encoding="utf-8") as checkpoint_file:
            return validate_checkpoint_shape(json.load(checkpoint_file))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Checkpoint JSON is malformed: {exc}") from exc


existing_files = [path for path in [CHECKPOINT_PATH, *OUTPUT_PATHS.values()] if path.exists()]
if CHECKPOINT_PATH.exists():
    mode = input("Checkpoint found. Enter R to resume or N to start a new dataset: ").strip().upper()
    if mode == "R":
        STATE = load_checkpoint()
        print(f"Resuming {len(STATE['records']):,} saved unique records.")
    elif mode == "N":
        if input("Type NEW to confirm replacing the checkpoint and eventual output files: ").strip() != "NEW":
            raise RuntimeError("New generation was not confirmed; existing files were left untouched.")
        STATE = new_generation_state()
        duplicate_detector = DuplicateDetector()
        save_checkpoint(STATE)
    else:
        raise RuntimeError("Choose R (resume) or N (new dataset).")
else:
    if existing_files:
        if input("Output files exist without a checkpoint. Type NEW to start and replace them after completion: ").strip() != "NEW":
            raise RuntimeError("New generation was not confirmed; existing files were left untouched.")
    STATE = new_generation_state()
    duplicate_detector = DuplicateDetector()
    save_checkpoint(STATE)

if duplicate_detector is None:
    duplicate_detector = DuplicateDetector(STATE["records"])


## 11. Live progress dashboard

Show overall progress, current category/batch, rejection counters, API requests, and per-category completion.

In [ ]:
def show_progress(state, progress_value=None):
    generated = len(state["records"])
    current_category = next(
        (category for category, quota in CATEGORY_QUOTAS.items()
         if state["category_counts"][category] < quota),
        "complete",
    )
    current_batch = min(60, generated // BATCH_SIZE + (1 if generated < TARGET_SIZE else 0))
    percent = generated * 100 / TARGET_SIZE
    print(
        f"\nBatch: {current_batch}/60 | Generated: {generated:,}/{TARGET_SIZE:,} "
        f"({percent:.1f}%) | Remaining: {TARGET_SIZE - generated:,}\n"
        f"Category: {current_category} | API requests: {state['api_requests']} | "
        f"Duplicates: {state['duplicate_rejections']} | Invalid: {state['invalid_records']}"
    )
    table = pd.DataFrame(
        [
            {
                "category": category,
                "generated": state["category_counts"][category],
                "target": quota,
                "remaining": quota - state["category_counts"][category],
            }
            for category, quota in CATEGORY_QUOTAS.items()
        ]
    )
    display(table)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(DRIVE_DIR / "generation.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,
)
LOGGER = logging.getLogger("sarcasm_generator")

if STATE["records"]:
    print(f"Loaded checkpoint with {len(STATE['records']):,} records.")
STATE = generate_dataset(STATE)
print("Generation loop finished; run the validation cell next.")


## 12. Dataset validation

This strict check reads saved JSONL and verifies exact size, schema, IDs, category quotas, unique normalized questions/pairs, valid levels, and non-empty strings.

In [ ]:
def validate_dataset(records, jsonl_path=None):
    errors = []
    if len(records) != TARGET_SIZE:
        errors.append(f"Expected {TARGET_SIZE} records; found {len(records)}.")
    expected_keys = REQUIRED_FIELDS
    for index, record in enumerate(records, start=1):
        if not isinstance(record, dict) or set(record) != expected_keys:
            errors.append(f"Record {index} has missing/extra fields.")
            continue
        if type(record["id"]) is not int or record["id"] != index:
            errors.append(f"Record {index} has ID {record['id']!r}, expected {index}.")
        if not isinstance(record.get("category"), str) or record["category"] not in CATEGORY_QUOTAS:
            errors.append(f"Record {index} has unknown category {record['category']!r}.")
        if type(record["sarcasm_level"]) is not int or record["sarcasm_level"] not in LEVELS:
            errors.append(f"Record {index} has invalid sarcasm level.")
        if not isinstance(record["question"], str) or not record["question"].strip():
            errors.append(f"Record {index} has an empty/invalid question.")
        if not isinstance(record["answer"], str) or not record["answer"].strip():
            errors.append(f"Record {index} has an empty/invalid answer.")

    category_counts = Counter(
        record.get("category") for record in records
        if isinstance(record, dict) and isinstance(record.get("category"), str)
    )
    for category, target in CATEGORY_QUOTAS.items():
        actual = category_counts.get(category, 0)
        if actual != target:
            errors.append(f"Category {category}: expected {target}, found {actual}.")
    question_keys = [normalize_text(record["question"]) for record in records if isinstance(record, dict) and isinstance(record.get("question"), str)]
    if len(question_keys) != len(set(question_keys)):
        errors.append("Duplicate normalized questions found.")
    pair_keys = [
        (normalize_text(record["question"]), normalize_text(record["answer"]))
        for record in records if isinstance(record, dict) and isinstance(record.get("question"), str) and isinstance(record.get("answer"), str)
    ]
    if len(pair_keys) != len(set(pair_keys)):
        errors.append("Duplicate normalized question-answer pairs found.")
    level_counts = Counter(
        record.get("sarcasm_level") for record in records
        if isinstance(record, dict) and type(record.get("sarcasm_level")) is int
    )
    if any(level_counts[level] == 0 for level in LEVELS):
        errors.append("One or more sarcasm levels are missing.")

    if jsonl_path is not None:
        parsed = []
        try:
            with Path(jsonl_path).open("r", encoding="utf-8") as source:
                for line_number, line in enumerate(source, start=1):
                    try:
                        parsed.append(json.loads(line))
                    except json.JSONDecodeError as exc:
                        errors.append(f"JSONL line {line_number} is malformed: {exc}.")
            if len(parsed) != len(records):
                errors.append(f"JSONL has {len(parsed)} parsed lines, expected {len(records)}.")
        except OSError as exc:
            errors.append(f"Unable to read JSONL file: {exc}")
    return errors


validation_errors = validate_dataset(STATE["records"])
if validation_errors:
    print("\\n".join(validation_errors))
    raise RuntimeError("Dataset validation failed. Checkpoint is retained; generation must be resumed/fixed.")
print("In-memory dataset validation passed.")


## 13. Export dataset

Write the original JSONL and CSV plus deterministic, category-stratified 90/10 chat-format training and validation JSONL. Source records remain unchanged.

In [ ]:
def export_dataset(records):
    records = sorted(records, key=lambda record: record["id"])
    with OUTPUT_PATHS["jsonl"].open("w", encoding="utf-8", newline="\n") as output:
        for record in records:
            output.write(json.dumps(record, ensure_ascii=False) + "\n")

    with OUTPUT_PATHS["csv"].open("w", encoding="utf-8-sig", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=["id", "category", "sarcasm_level", "question", "answer"])
        writer.writeheader()
        writer.writerows(records)

    rng = random.Random(RANDOM_SEED)
    by_category = defaultdict(list)
    for record in records:
        by_category[record["category"]].append(record)
    train_records, validation_records = [], []
    for category, category_records in by_category.items():
        category_records = list(category_records)
        rng.shuffle(category_records)
        validation_count = round(len(category_records) * 0.10)
        validation_records.extend(category_records[:validation_count])
        train_records.extend(category_records[validation_count:])
    train_records.sort(key=lambda record: record["id"])
    validation_records.sort(key=lambda record: record["id"])

    def write_chat_jsonl(path, split_records):
        with path.open("w", encoding="utf-8", newline="\n") as output:
            for record in split_records:
                row = {
                    "messages": [
                        {"role": "user", "content": record["question"]},
                        {"role": "assistant", "content": record["answer"]},
                    ]
                }
                output.write(json.dumps(row, ensure_ascii=False) + "\n")

    write_chat_jsonl(OUTPUT_PATHS["train"], train_records)
    write_chat_jsonl(OUTPUT_PATHS["validation"], validation_records)
    if len(train_records) != 5_400 or len(validation_records) != 600:
        raise RuntimeError(
            f"Expected 5400/600 train/validation records; got "
            f"{len(train_records)}/{len(validation_records)}."
        )
    train_questions = {normalize_text(record["question"]) for record in train_records}
    validation_questions = {normalize_text(record["question"]) for record in validation_records}
    if train_questions & validation_questions:
        raise RuntimeError("Question leakage detected between training and validation sets.")
    return train_records, validation_records


train_records, validation_records = export_dataset(STATE["records"])
disk_errors = validate_dataset(STATE["records"], OUTPUT_PATHS["jsonl"])
if disk_errors:
    raise RuntimeError("Export validation failed:\\n" + "\\n".join(disk_errors))
print("Original JSONL, CSV, training JSONL, and validation JSONL were exported and checked.")


## 14. Download buttons

Use Colab's download helper to download individual exports. The files remain in Drive regardless of downloads.

In [ ]:
from google.colab import files

def download_dataset_file(name):
    path = OUTPUT_PATHS[name]
    if not path.exists():
        raise FileNotFoundError(f"Export does not exist: {path}")
    files.download(str(path))

print("Run one of these to download a file:")
print("download_dataset_file('jsonl')")
print("download_dataset_file('csv')")
print("download_dataset_file('train')")
print("download_dataset_file('validation')")
print("download_dataset_file('report')")


## 15. Final dataset report

Print validated record counts, category and sarcasm-level distributions, rejection totals, API requests, split sizes, file sizes, and Drive paths.

In [ ]:
def build_report(state, train_rows, validation_rows):
    records = state["records"]
    category_counts = Counter(record["category"] for record in records)
    level_counts = Counter(record["sarcasm_level"] for record in records)
    lines = [
        "SARCASTIC CONVERSATIONAL DATASET GENERATION REPORT",
        f"Generated at: {datetime.now(timezone.utc).isoformat()}",
        f"Model: {MODEL}",
        f"Total examples: {len(records)}",
        f"Unique examples: {len({(normalize_text(r['question']), normalize_text(r['answer'])) for r in records})}",
        f"Duplicate rejections: {state['duplicate_rejections']}",
        f"Invalid records: {state['invalid_records']}",
        f"Surplus records: {state.get('surplus_records', 0)}",
        f"API requests: {state['api_requests']}",
        f"Training records: {len(train_rows)}",
        f"Validation records: {len(validation_rows)}",
        "",
        "Category distribution:",
    ]
    lines.extend(f"  {category}: {category_counts.get(category, 0)} / {quota}" for category, quota in CATEGORY_QUOTAS.items())
    lines.append("Sarcasm-level distribution:")
    lines.extend(f"  Level {level}: {level_counts.get(level, 0)}" for level in LEVELS)
    lines.append("Output files:")
    for name, path in OUTPUT_PATHS.items():
        size = path.stat().st_size if path.exists() else 0
        lines.append(f"  {name}: {path} ({size:,} bytes)")
    lines.append(f"Checkpoint: {CHECKPOINT_PATH}")
    return "\n".join(lines)


final_errors = validate_dataset(STATE["records"], OUTPUT_PATHS["jsonl"])
if final_errors:
    raise RuntimeError("Final validation failed:\n" + "\n".join(final_errors))
report_text = build_report(STATE, train_records, validation_records)
OUTPUT_PATHS["report"].write_text(report_text + "\n", encoding="utf-8")
report_text = build_report(STATE, train_records, validation_records)
OUTPUT_PATHS["report"].write_text(report_text + "\n", encoding="utf-8")
print(report_text)
print("\nGeneration and validation completed successfully.")
